# Setup

In [3]:
import os
import json

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "ibm-granite/granite-4.1-8b"

In [4]:
import langchain_openrouter

from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage
from langchain_core.messages import AIMessage

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from pydantic import BaseModel, Field

/tmp/ipykernel_2210/791403691.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
## list all the parameters that can be used to create a chat model
help(langchain_openrouter.ChatOpenRouter)

# Create Model

In [ ]:
def llm_model(params=None):

    # 1. Define sensible defaults
    config = {
        "model": OPENROUTER_MODEL,
        "api_key": OPENROUTER_API_KEY,
        "base_url": OPENROUTER_DEFAULT_BASE_URL,
        "temperature": 0.5,
        "max_tokens": 256,
        "max_completion_tokens": 128
    }
    
    if params:
        config.update(params)
        
    # 3. Initialize the model
    model = langchain_openrouter.ChatOpenRouter(
        model=config["model"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        max_completion_tokens=config["max_completion_tokens"]
    )

    return model

def llm_model_response(prompt_text, params=None):
            
    # 3. Initialize the model
    model = llm_model(params)

    response = model.invoke(prompt_text)

    return response

# Langchain Concepts

## Chat Message

In [ ]:
from prompt_toolkit import prompt


OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)


In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        # can also exclude SystemMessage and it will default to a helpful assistant
        #SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

# creative
params_creative = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

# precise
params_precise = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,  
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.2,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

model_creative = llm_model(params=params_creative)
model_precise = llm_model(params=params_precise)

prompts = [
    "Write a short poem about artificial intelligence",
    "What are the key components of a neural network?",
    "List 5 tips for effective time management"
]

for prompt in prompts:
    response_creative = model_creative.invoke(prompt)
    response_precise = model_precise.invoke(prompt)

    print(f"Prompt: {prompt}")
    print(f"Creative Response: {response_creative.text}")
    print(f"Precise Response: {response_precise.text}")
    print("-" * 50)



## Prompt Templates

### String prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")

input = {"adjective": "funny", "topic": "cats"} 

prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Chat prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a list of message tuples
# Each tuple contains a role ("system" or "user") and the message content
# The system message sets the behavior of the assistant
# The user message includes a variable placeholder {topic} that will be replaced later
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

# Create a dictionary with the variable to be inserted into the template
# The key "topic" matches the placeholder name in the user message
input = {"topic": "cats"}

# Format the chat template with our input values
# This replaces {topic} with "cats" in the user message
# The result will be a formatted chat message structure ready to be sent to a model
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Messages Placeholder

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# The system message sets the behavior for the assistant
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")  
])

# Create an input dictionary where the key matches the MessagesPlaceholder name
# The value is a list of message objects that will replace the placeholder
# Here we're adding a single HumanMessage asking about the day after Tuesday
input = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

# Format the chat template with our input dictionary
# This replaces the MessagesPlaceholder with the HumanMessage in our input
# The result will be a formatted chat structure with a system message and our human message
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

## Output Parsers

### JSON parser

In [ ]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

# Get the formatting instructions for the output parser
# This generates guidance text that tells the LLM how to format its response
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"query": joke_query})

print(json.dumps(response, indent=4))

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = JsonOutputParser()

format_instructions = """RESPONSE FORMAT INSTRUCTIONS: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant. Task: Generate info about the movie "{movie_name}" in JSON format. {format_instructions}""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

movie_chain = prompt_template | model | output_parser
movie_name = "Inception"
response = movie_chain.invoke({"movie_name": movie_name})

print("Parsed result:")
print(f"Title: {response['title']}")
print(f"Director: {response['director']}")
print(f"Year: {response['year']}")
print(f"Genre: {response['genre']}")




### Comma-separated list parser

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()

# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{subject}\n",
    input_variables=["subject"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"subject": "ice cream flavors"})

print(response)

## Documents

### Document Object

In [ ]:
Document( 
    page_content="""Python is an interpreted high-level general-purpose programming language.
                    Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
    metadata= {
        'my_document_id' : 234234,                      # Unique identifier for this document
        'my_document_source' : "About Python",          # Source or title information
        'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
    }
)

### Document Loaders

Document loaders in LangChain are designed to load documents from a variety of sources; for instance, loading a PDF file and having the LLM read the PDF file using LangChain.

LangChain offers over 100 distinct document loaders, along with integrations with other major providers, such as AirByte and Unstructured. These integrations enable loading of all kinds of documents (HTML, PDF, code) from various locations including private Amazon S3 buckets, as well as from public websites).

In [ ]:
## PDF Loader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

document = loader.load()

## print formatted json metadata of the first page of the document
print(json.dumps(document[0].metadata, indent=4))
print(document[0].page_content)
##print(document[0].page_content[:1000])


## Splitters
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))
print(chunks[0].page_content)


In [ ]:
### Websire Loader
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

web_data = loader.load()

print(web_data[0].page_content[:1000])

## Splitters
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
texts = text_splitter.split_documents(web_data)
print(len(texts))

In [ ]:
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30, separators=["\n\n", "\n", " ", ""])

chunks_pdf = splitter_1.split_documents(pdf_document)
chunks_web = splitter_2.split_documents(web_document)

def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

display_document_stats(chunks_pdf, "PDF Document")
display_document_stats(chunks_web, "Web Document")


### Embedding

Embedding models are specifically designed to interface with text embeddings.

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.


In [ ]:
OPENROUTER_MODEL = "openai/text-embedding-3-small"

## PDF Loader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

document = loader.load()

## print formatted json metadata of the first page of the document
##print(json.dumps(document[0].metadata, indent=4))
##print(document[0].page_content)
##print(document[0].page_content[:1000])


## Splitters
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100, separator="\n")
chunks = text_splitter.split_documents(document)
##print(len(chunks))
##print(chunks[0].page_content)

from openrouter import OpenRouter

texts = [chunk.page_content for chunk in chunks]

with OpenRouter(api_key=OPENROUTER_API_KEY, server_url=OPENROUTER_DEFAULT_BASE_URL) as openrouter:
    result = openrouter.embeddings.generate(
        model=OPENROUTER_MODEL,
        input=texts
    )
    
print(result.data[0].embedding[:100])  # Print the first 100 dimensions of the first embedding vector


### Embbeding, Batches

In [ ]:
# Usar OpenRouter e n\ao as libs de langchain, langchain_openrouter n~\ao suporta embbedings
from openrouter import OpenRouter # Ensure this matches your local SDK package structure

OPENROUTER_MODEL = "openai/text-embedding-3-small"

# 1. Load document
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # Increased for better contextual meaning
    chunk_overlap=120,    # Generous overlap to keep context intact
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
texts = [chunk.page_content for chunk in chunks]

# 3. Batching API calls (Handling max 100 texts at a time)
batch_size = 100
all_embeddings = []

with OpenRouter(api_key=OPENROUTER_API_KEY, server_url=OPENROUTER_DEFAULT_BASE_URL) as openrouter:
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = openrouter.embeddings.generate(
            model=OPENROUTER_MODEL,
            input=batch,
            dimensions=1024
        )
        all_embeddings.extend([item.embedding for item in result.data])

print(f"Successfully generated {len(all_embeddings)} embeddings.")
print(f"Sample of first embedding vector: {all_embeddings[0][:5]}...")

In [ ]:
#usar langchain_openai

OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
texts = [chunk.page_content for chunk in chunks]

# 3. Native LangChain Embeddings configuration
# LangChain automatically handles chunk batching behind the scenes!
embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

# 4. Manual loop batching
batch_size = 100
all_embeddings = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]
    print(f"Processing batch {i // batch_size + 1}: items {i} to {i + len(batch_texts)}")
    
    # Generate embeddings just for this specific batch
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    
    # Accumulate results
    all_embeddings.extend(batch_embeddings)

print(f"Successfully generated {len(all_embeddings)} embeddings.")
print(f"Sample of first embedding vector: {all_embeddings[0][:5]}...")

Processing batch 1: items 0 to 40
Successfully generated 40 embeddings.
Sample of first embedding vector: [0.021759033203125, -0.038177490234375, 0.073486328125, 0.06817626953125, -0.0189056396484375]...


### Vectorstores, Chroma

In [ ]:
# Fazer o mesmo com ChromaDb
OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
#texts = [chunk.page_content for chunk in chunks]

# 3. Native LangChain Embeddings configuration
# LangChain automatically handles chunk batching behind the scenes!
embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

docsearch = Chroma.from_documents(chunks, embeddings_model)
query = "Langchain"
docs = docsearch.similarity_search(query)
print(docs[0].page_content)